<img src="https://raw.githubusercontent.com/mjcowley/astrolabs/main/Images/qarg_bw_dark.png" alt="QUT Astrophysics" width="350"/>

# AstroLab C9: Galaxies and Their Monsters

**PVB220 Cosmology** | Dr Michael Cowley | michael.cowley@qut.edu.au

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjcowley/astrolabs/blob/main/AstroLabC9.ipynb)

## Introduction

In the last two lectures we met the galaxies of the Hubble tuning fork, and the supermassive black holes that power active galactic nuclei (AGN) at their centres. In this lab you'll work with the same kinds of data professional astronomers use to study them.

You'll:

1. **Classify real galaxies** on the Hubble tuning fork and measure how squashed an elliptical is
2. **Read galaxy spectra** and turn them into **colours**
3. Use a real survey to find **red and dead** galaxies with the **UVJ diagram**, and watch them appear over cosmic time
4. Redo the lecture's **3C 273** worked examples with its real spectrum: redshift, distance, brightness and **Eddington limit**
5. Measure how fast gas is moving around a black hole from **broad and narrow lines**
6. Find AGN with **X-rays** and ask where they live: are black holes switching off star formation?

Have Canvas open alongside this notebook and answer each 🎯 **Canvas checkpoint** as you go.

▶️ Run the cell below first.

In [ ]:
#@title ▶️ Run me first: setup
import os
import urllib.request

# Download the latest helper file (keep the existing copy if we're offline)
try:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/mjcowley/astrolabs/main/astrolabs.py",
        "astrolabs.py")
except Exception:
    if not os.path.exists("astrolabs.py"):
        raise

import numpy as np                 # numbers and arrays
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # plotting
import matplotlib.image as mpimg   # reading images
import astropy.units as u          # physical units
from astropy.constants import c    # the speed of light
import importlib
import astrolabs as al             # helper functions for these labs
importlib.reload(al)               # use the version just downloaded

al.setup()

---
# Part 1: The Hubble Tuning Fork

Edwin Hubble sorted galaxies by their **shape** (morphology): ellipticals (E), lenticulars (S0), spirals (S), barred spirals (SB) and irregulars (Irr).

Let's load seven real galaxies imaged by the DESI Legacy Imaging Surveys. We store them in a **dictionary**, just like the distances in AstroLab C7.

In [ ]:
galaxy_names = ["M87", "NGC4473", "NGC3115", "M104", "M74", "NGC1300", "NGC1427A"]

images = {}
for name in galaxy_names:
    images[name] = mpimg.imread(al.get_data(f"C9/images/{name}.jpg"))

print("Loaded", len(images), "galaxy images")
print("Each image is an array with shape", images["M87"].shape, "(pixels high, pixels wide, colours)")

> 🔍 **Peek inside:** `for name in galaxy_names:` is a **loop** that runs the indented line once for each galaxy. `f"C9/images/{name}.jpg"` builds each file name by slotting the galaxy's name into the text.

▶️ Run the cell below. Pick each galaxy from the menu and classify it. The widget tells you if you're right.

In [ ]:
al.galaxy_classifier(images)

> 🎯 **Canvas checkpoint Q1:** One of these galaxies looks like a very flat elliptical, but is really a lenticular (S0) seen edge-on. Which one?

## Measuring an elliptical

The lecture classified ellipticals using

$$E = 10\left(1 - \frac{b}{a}\right)$$

where $a$ and $b$ are the semi-major and semi-minor axes. A perfectly round galaxy ($b/a = 1$) is an E0, and the flattest are about E7.

▶️ Run the cell below and fit the cyan ellipse to the outer, faint glow of **M87**: first set the size, then the axis ratio $b/a$, then the angle.

In [ ]:
al.ellipse_fitter(images["M87"], "M87")

👉 Now fit **NGC 4473**.

In [ ]:
al.ellipse_fitter(images["NGC4473"], "NGC 4473")

In [ ]:
# 👉 Type in the axis ratio b/a you found for NGC 4473, then run
axis_ratio = None

if axis_ratio is not None:
    E = 10 * (1 - axis_ratio)
    print(f"NGC 4473: E = {E:.1f}, so it's an E{int(E + 0.5)}")
    al.check("C9_E_ngc4473", E)

> 🎯 **Canvas checkpoint Q2:** Which Hubble class best describes **M87**?
>
> 🎯 **Canvas checkpoint Q3:** Which Hubble class best describes **NGC 4473**?

💡 Different people get slightly different answers, especially as the bright centre and the faint outskirts can have different shapes. That's why professional astronomers measure shapes with computer code, but the idea is exactly the same.

---
# Part 2: Galaxy Spectra and Colours

In the lecture we saw that the **Andromeda Galaxy** has a spectrum like a star (a continuum with absorption lines), while the **Orion Nebula** shines mostly in bright emission lines. Galaxies that are forming lots of new stars have both.

These spectra come from an atlas of nearby galaxies, and include several from the gallery above. Each spectrum has been scaled to the same brightness at 0.55 μm (green light).

In [ ]:
spectra = pd.read_csv(al.get_data("C9/galaxy_spectra.csv"))

print("Galaxies:", list(spectra["galaxy"].unique()))
spectra.head()

In [ ]:
# 👉 Try adding "NGC 4473 (elliptical)" or "M104 Sombrero (lenticular/Sa)" to the list
to_plot = ["M87 (elliptical)", "M74 (spiral)", "NGC 3310 (starburst)"]

fig, ax = plt.subplots(figsize=(11, 5))
for name in to_plot:
    one = spectra[spectra["galaxy"] == name]
    ax.plot(one["wavelength_um"], one["flux"], lw=1, label=name)

ax.set_xlim(0.3, 0.9)
ax.set_ylim(0, 3)
ax.set_xlabel("Wavelength (micrometres)")
ax.set_ylabel("Brightness (scaled at 0.55 μm)")
ax.legend()
plt.show()

> 🎯 **Canvas checkpoint Q4:** Which galaxy's spectrum has the strongest **emission lines** (sharp spikes), a sign of lots of hot young stars lighting up gas?

## From spectra to colours

Most galaxies are too faint for a spectrum, so astronomers measure their brightness through a few **filters** instead and compare them. Comparing the brightness in two filters gives a **colour**. Here we'll use three standard filters: **U** (ultraviolet, 0.365 μm), **V** (visual, 0.55 μm) and **J** (near-infrared, 1.22 μm).

Colours use the **magnitude** scale from PVB210, where *bigger numbers mean fainter*. So a **large U − V** means the galaxy is faint in the ultraviolet compared with green light: it's **red**.

Here's a function that measures a galaxy's magnitude through a simple filter centred on a given wavelength:

In [ ]:
def magnitude(spectrum, centre_um, width_um):
    # Average brightness through a simple "top-hat" filter, as a magnitude
    inside = (spectrum["wavelength_um"] > centre_um - width_um / 2) & \
             (spectrum["wavelength_um"] < centre_um + width_um / 2)
    # Convert brightness per wavelength to per frequency (the AB system), then to magnitudes
    flux_nu = spectrum["flux"][inside] * spectrum["wavelength_um"][inside] ** 2
    return -2.5 * np.log10(flux_nu.mean())

for name in spectra["galaxy"].unique():
    one = spectra[spectra["galaxy"] == name]
    U = magnitude(one, 0.365, 0.06)
    V = magnitude(one, 0.551, 0.09)
    J = magnitude(one, 1.22, 0.20)
    print(f"{name:32s}  U − V = {U - V:.2f}   V − J = {V - J:.2f}")

> 🎯 **Canvas checkpoint Q5:** Which galaxy is the **bluest** (smallest U − V)?

Just as the lecture said, ellipticals are **red** (old stars, no new star formation) and starbursts are **blue** (full of young, hot, massive stars).

## A real galaxy survey: ZFOURGE

Now let's look at tens of thousands of galaxies from the **FourStar Galaxy Evolution Survey (ZFOURGE)**, which imaged a small patch of sky called the Chandra Deep Field South in about 40 filters. QUT researchers, including your lecturer, have used ZFOURGE to study how galaxies and their black holes grow.

For every galaxy, the survey team estimated its redshift, its **stellar mass** (in solar masses, as a power of ten), its star formation rate and its colours.

In [ ]:
galaxies = pd.read_csv(al.get_data("C9/zfourge_cdfs_galaxies.csv"))

print("Number of galaxies:", len(galaxies))
galaxies.head()

`log_mass` is the power of ten: a galaxy with `log_mass = 10.5` has $10^{10.5} \approx 30$ billion solar masses of stars. The Milky Way's is about 10.8.

## Selecting galaxies with conditions

To study a fair sample, we'll keep galaxies that are fairly massive and have measured colours, between redshift 0.5 and 2.5. In `pandas` we write each **condition** in brackets and combine them with `&` ("and").

In [ ]:
sample = galaxies[(galaxies["log_mass"] > 9.5) &
                  (galaxies["z"] > 0.5) & (galaxies["z"] < 2.5) &
                  galaxies["U_minus_V"].notna() & galaxies["V_minus_J"].notna()]

print("Galaxies in our sample:", len(sample))

## The colour–mass diagram

This is the same kind of plot as the lecture's "Colours of Galaxies" slide.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(sample["log_mass"], sample["U_minus_V"], s=4, alpha=0.4, color=al.COLOURS["grey"])
ax.set_xlabel("Stellar mass, log(M / M☉)")
ax.set_ylabel("U − V colour")
ax.set_ylim(0, 2.6)
ax.set_title("Redder galaxies are higher up")
plt.show()

Look for the two clumps: a **red sequence** of massive red galaxies at the top right, and a **blue cloud** below it, with a thinner **green valley** in between.

## The UVJ diagram

There's a catch. A galaxy can look red for two reasons: its stars are **old**, or it's **dusty** and still forming stars behind the dust. Adding a second colour, **V − J**, separates the two. Old galaxies are red in U − V but not too red in V − J; dusty star-forming galaxies are red in both.

Astronomers draw a box on this **UVJ diagram**: galaxies inside it are **quiescent** ("red and dead", no longer forming stars). The box is defined by three conditions:

* U − V > 1.3
* V − J < 1.6
* U − V > 0.88 × (V − J) + 0.69

In [ ]:
uv = sample["U_minus_V"]
vj = sample["V_minus_J"]

# 👉 These three conditions define the quiescent box
in_box = (uv > 1.3) & (vj < 1.6) & (uv > 0.88 * vj + 0.69)

quiescent = sample[in_box]
star_forming = sample[~in_box]          # ~ means "not"

print(f"Quiescent galaxies:    {len(quiescent)}")
print(f"Star-forming galaxies: {len(star_forming)}")
print(f"Quiescent fraction:    {len(quiescent) / len(sample):.0%}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(star_forming["V_minus_J"], star_forming["U_minus_V"], s=4, alpha=0.4, color=al.COLOURS["blue"])
ax.scatter(quiescent["V_minus_J"], quiescent["U_minus_V"], s=4, alpha=0.5, color=al.COLOURS["red"])
al.draw_uvj_box(ax)
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.2, 2.6)
ax.set_xlabel("V − J colour")
ax.set_ylabel("U − V colour")
plt.show()

> 🔍 **Peek inside:** `al.is_quiescent` in `astrolabs.py` uses exactly the same three conditions, so we can reuse them later.

## Watching galaxies die

How does the number of quiescent galaxies change as the Universe ages? ▶️ Run the cell below and move the redshift range from **2.0 to 2.5** (early, when the Universe was about 3 billion years old) down to **0.5 to 1.0** (about 6 to 9 billion years old).

In [ ]:
al.uvj_explorer(galaxies)

> 🎯 **Canvas checkpoint Q6:** Roughly what percentage of galaxies are quiescent at 0.5 < z < 1.0?
>
> 🎯 **Canvas checkpoint Q7:** How does the quiescent fraction change as the Universe gets older?

Something is switching off star formation in galaxies over cosmic time. One suspect is the supermassive black hole at the centre. Let's meet one.

---
# Part 3: The Quasar 3C 273

In the lecture we met **3C 273**, the 273rd object in the Third Cambridge Catalogue of radio sources. It looks like a faint star, but its spectrum was unlike any star's. Here is its real spectrum.

In [ ]:
agn_spectra = pd.read_csv(al.get_data("C9/agn_spectra.csv"))
quasar = agn_spectra[agn_spectra["object"] == "3C 273 (quasar)"]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(quasar["wavelength_A"], quasar["flux"], color="k", lw=1)
ax.set_xlabel("Observed wavelength (Å)")
ax.set_ylabel("Brightness (scaled)")
ax.set_title("The spectrum of 3C 273")
plt.show()

The broad bumps are hydrogen emission lines, but they're in the wrong places! ▶️ Run the cell below and slide the redshift until the dashed lines sit on the peaks. Start with **Hα**, the tallest line.

In [ ]:
rest_lines = {"Hγ": 4340.5, "Hβ": 4861.3, "[O III]": 5006.8, "Hα": 6562.8}

al.line_explorer(quasar["wavelength_A"], quasar["flux"], rest_lines, "3C 273", zmax=0.3)

> 🎯 **Canvas checkpoint Q8:** What is the redshift of 3C 273?

## Redoing the lecture's worked examples

Now we'll repeat the three worked examples from the lecture in Python. Enter your redshift and run each cell.

In [ ]:
# 👉 Your redshift for 3C 273
z = None

if z is not None:
    H0 = 71 * u.km / u.s / u.Mpc        # the Hubble constant used in the lecture
    v = z * c                            # recession speed, v = cz
    d = v / H0                           # Hubble's law, d = v / H0

    print(f"Recession speed: v = {v.to(u.km / u.s):,.0f}")
    print(f"Distance:        d = {d.to(u.Mpc):,.0f}")
    print(f"               = {d.to(u.lyr).value / 1e9:.1f} billion light-years")
    al.check("C9_distance_mpc", d.to(u.Mpc))

> 🎯 **Canvas checkpoint Q9:** How far away is 3C 273?

### How bright is it?

3C 273 has an absolute magnitude of −26.7. The whole Andromeda Galaxy has −21.5. The brightness ratio is $B_2/B_1 = 100^{(m_1 - m_2)/5}$.

In [ ]:
M_andromeda = -21.5
M_3c273 = -26.7

ratio = 100 ** ((M_andromeda - M_3c273) / 5)
print(f"3C 273 is {ratio:.0f} times brighter than the entire Andromeda Galaxy")
al.check("C9_brightness_ratio", ratio)

> 🎯 **Canvas checkpoint Q10:** How many times brighter than the whole Andromeda Galaxy is 3C 273?

And all that light comes from a region about the size of our Solar System.

### The Eddington limit

If a black hole shines too brightly, its radiation pushes away the gas that feeds it. For it to keep shining, the black hole must be at least the **Eddington mass**:

$$M_{\rm Edd} \approx 3.1 \times 10^{-5} \, \frac{L}{L_\odot} \; M_\odot$$

and it must swallow gas at about the **Eddington rate**, $\dot M_{\rm Edd} \approx 2.4 \times 10^{-8} \, M \; M_\odot\,{\rm yr}^{-1}$.

In [ ]:
M_sun = 4.77                                  # absolute magnitude of the Sun

luminosity = 100 ** ((M_sun - M_3c273) / 5)   # in solar luminosities
eddington_mass = 3.1e-5 * luminosity          # in solar masses
eddington_rate = 2.4e-8 * eddington_mass      # in solar masses per year

print(f"Luminosity:     {luminosity:.2e} L☉")
print(f"Eddington mass: {eddington_mass:.2e} M☉")
print(f"Eddington rate: {eddington_rate:.1f} M☉ per year")
al.check("C9_eddington_mass", eddington_mass)

> 🎯 **Canvas checkpoint Q11:** What is the minimum mass of the black hole powering 3C 273?

To keep shining, 3C 273's black hole has to swallow about **three Suns' worth of gas every year**.

---
# Part 4: Broad and Narrow Lines

In the unified model of AGN, gas clouds close to the black hole (the **broad-line region**) whirl around at thousands of km/s, while clouds further out (the **narrow-line region**) move more slowly. Because of the **Doppler effect**, fast-moving gas spreads a spectral line out over a wider range of wavelengths:

$$\frac{\Delta \lambda}{\lambda} = \frac{v}{c}$$

Let's compare the **Seyfert 1** galaxy NGC 4151 with the **Seyfert 2** galaxy NGC 1068.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for ax, name in zip(axes, ["NGC 4151 (Seyfert 1)", "NGC 1068 (Seyfert 2)"]):
    one = agn_spectra[agn_spectra["object"] == name]
    ax.plot(one["wavelength_A"], one["flux"], color="k", lw=1)
    ax.axvline(6563, color=al.COLOURS["red"], ls="--", lw=1)
    ax.set_title(name)
    ax.set_ylabel("Brightness (scaled)")
axes[1].set_xlabel("Wavelength (Å)")
plt.tight_layout()
plt.show()

The red dashed line marks **Hα**. ▶️ Run the cell below. For each galaxy, slide the gas speed until the orange curve has the same width as the Hα line.

In [ ]:
al.line_width_explorer(agn_spectra)

> 🎯 **Canvas checkpoint Q12:** Roughly how fast is the gas producing the broad Hα line in NGC 4151 (Seyfert 1) moving?
>
> 🎯 **Canvas checkpoint Q13:** According to the unified model, why doesn't NGC 1068 (Seyfert 2) show broad lines?

💡 Try the [AGN Unified Model demo](https://mjcowley.github.io/astro_demos/agn/) to see how the viewing angle changes what we see.

---
# Part 5: Where Do AGN Live?

## Finding AGN with X-rays

Accretion discs around black holes shine brightly in **X-rays**, far more than ordinary galaxies. The **Chandra X-ray Observatory** stared at the Chandra Deep Field South for 7 million seconds (about 81 days!), the deepest X-ray image ever taken. Here are the sources it found, each classified by the Chandra team as an AGN or an ordinary galaxy.

In [ ]:
xray = pd.read_csv(al.get_data("C9/chandra_cdfs_7ms.csv"))

print("X-ray sources:", len(xray))
print(xray["type"].value_counts())
xray.head()

In [ ]:
# 👉 The lecture's AGN threshold: X-ray luminosity above 10^42 erg/s
threshold = 42

fig, ax = plt.subplots(figsize=(9, 5.5))
for kind, colour in [("Galaxy", al.COLOURS["grey"]), ("AGN", al.COLOURS["blue"])]:
    one = xray[xray["type"] == kind]
    ax.scatter(one["z"], one["log_Lx"], s=10, alpha=0.6, color=colour, label=kind)
ax.axhline(threshold, color=al.COLOURS["red"], ls="--", label="Lx = 10$^{42}$ erg/s")
ax.set_xlabel("Redshift")
ax.set_ylabel("X-ray luminosity, log(Lx / erg s$^{-1}$)")
ax.set_xlim(0, 5)
ax.legend()
plt.show()

bright = xray[xray["log_Lx"] > threshold]
print(f"Sources above the threshold: {len(bright)}")
print(f"Of those, classified as AGN: {(bright['type'] == 'AGN').mean():.0%}")

> 🎯 **Canvas checkpoint Q14:** What percentage of X-ray sources brighter than $10^{42}$ erg/s are classified as AGN?

💡 Notice that the faintest sources are only detected nearby. Distant objects must be very luminous for us to see them at all. Keep this in mind for the stretch activity!

## Matching AGN to their host galaxies

Which galaxies host these AGN? The ZFOURGE team **cross-matched** the Chandra catalogue with their own: they found which X-ray source sits at the position of each galaxy (Cowley et al. 2016). The result is a small table linking each galaxy's `id` to whether it hosts an X-ray AGN.

To combine two tables that share a column, we **merge** them. This is the Python version of the cross-matching you may have done in TOPCAT, and you'll do your own in AstroLab C11.

In [ ]:
matches = pd.read_csv(al.get_data("C9/zfourge_xray_matches.csv"))

hosts = sample.merge(matches, on="id")          # keep galaxies that appear in both tables
agn_hosts = hosts[hosts["xray_agn"] == 1]

print("X-ray AGN host galaxies in our sample:", len(agn_hosts))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(sample["V_minus_J"], sample["U_minus_V"], s=3, alpha=0.25, color=al.COLOURS["grey"], label="All galaxies")
ax.scatter(agn_hosts["V_minus_J"], agn_hosts["U_minus_V"], s=40, facecolor="none",
           edgecolor=al.COLOURS["orange"], lw=1.5, label="X-ray AGN hosts")
al.draw_uvj_box(ax)
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.2, 2.6)
ax.set_xlabel("V − J colour")
ax.set_ylabel("U − V colour")
ax.legend(loc="lower right")
plt.show()

# How many sit close to the edge of the box: the "green valley" between the two populations?
def near_boundary(d):
    return (abs(d["U_minus_V"] - (0.88 * d["V_minus_J"] + 0.69)) < 0.2) & \
           (d["U_minus_V"] > 1.0) & (d["V_minus_J"] < 1.6)

print(f"All galaxies near the boundary:     {near_boundary(sample).mean():.0%}")
print(f"AGN host galaxies near the boundary: {near_boundary(agn_hosts).mean():.0%}")

> 🎯 **Canvas checkpoint Q15:** Compared with all galaxies, are AGN hosts more or less likely to sit near the boundary between star-forming and quiescent galaxies?

AGN hosts are found **everywhere** on the diagram, but they're more common in the **green valley**, the in-between region where galaxies are thought to be shutting down star formation. That's a hint that AGN may be involved in **quenching**, but it isn't proof: as the lecture showed, some studies find AGN suppress star formation (negative feedback) while others find they go hand in hand (positive feedback). This is an active research question, including here at QUT!

---
# What's Coming Up

| Lab | Week | What you'll do |
| --- | :-: | --- |
| C7 | 2 | Python toolkit |
| C8 | 4 | The Big Bang and the CMB |
| **C9** | 6 | Galaxies and their black holes (this lab!) |
| **C10** | 8 | Gravitational waves: listen to colliding black holes |
| **C11** | 10 | Cosmology surveys: cross-matching catalogues and scaling relations |

### Sparkler ideas from this lab

* Classifying galaxies: what can shape tell us, and where does it fail?
* Red and dead: how quiescent galaxies appear over cosmic time
* 3C 273: the quasar that looked like a star
* Do black holes switch off star formation? AGN in the green valley

**Well done, you've finished AstroLab C9!** Make sure you've answered all the 🎯 Canvas checkpoints.

---
# 🚀 Stretch Activity: A Trap in the Data

*Optional, if you have time left.*

In the lecture's worked example on galaxy evolution, we looked at a plot of galaxy mass against redshift. Let's make it with real data.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(galaxies["z"], galaxies["log_mass"], s=1, alpha=0.2, color=al.COLOURS["blue"])
ax.set_xlim(0, 4)
ax.set_ylim(6, 12)
ax.set_xlabel("Redshift")
ax.set_ylabel("Stellar mass, log(M / M☉)")
plt.show()

At first glance it looks as if galaxies were **more massive** in the past, since there are almost no small galaxies at high redshift. But think about what the telescope can actually see.

In [ ]:
nearby = galaxies[galaxies["z"] < 0.5]
distant = galaxies[galaxies["z"] > 2.0]

print(f"Nearby  (z < 0.5): {(nearby['log_mass'] < 8).mean():.0%} of galaxies are below 10^8 M☉")
print(f"Distant (z > 2.0): {(distant['log_mass'] < 8).mean():.0%} of galaxies are below 10^8 M☉")

Small galaxies are faint, and distant galaxies are fainter still. Beyond a certain redshift, low-mass galaxies are simply **too faint to detect**, so they vanish from our catalogue even though they're really there. This **selection effect** (called **Malmquist bias** for distance surveys) creates the curved lower edge in the plot.

🚀 **Extra challenge:** compare galaxies above $10^{10}\,M_\odot$ only, which are bright enough to see at all redshifts. Copy the cell above and change the condition. Does the trend with redshift still look the same?

---
### Data and image credits

* Galaxy images: DESI Legacy Imaging Surveys DR9 and DR10 (Dey et al. 2019, AJ, 157, 168); M104 from the Sloan Digital Sky Survey via the Legacy Survey viewer
* Galaxy spectra: Brown et al. 2014, ApJS, 212, 18, via the STScI reference atlases
* AGN spectra: Brown et al. 2019, MNRAS, 489, 3351 (AGN SED Atlas, MAST, doi:10.17909/t9-3dbt-8734)
* ZFOURGE catalogue: Straatman et al. 2016, ApJ, 830, 51, via VizieR. Rest-frame U, V and J colours were estimated for this lab by interpolating each galaxy's photometry, so they're approximate
* X-ray AGN matches: Cowley et al. 2016, MNRAS, 457, 629
* Chandra Deep Field South 7 Ms catalogue: Luo et al. 2017, ApJS, 228, 2, via VizieR
* UVJ selection: Williams et al. 2009, ApJ, 691, 1879